<a href="https://colab.research.google.com/github/SohailVibeCoder/IB9AU---GenAI/blob/main/Task18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name:** Sohail Essajee (5757504)


TVAE produced more realistic synthetic data than CTGAN, especially in keeping the fraud ratio close to the original.

Models trained on synthetic data didn't perform as well as those trained on real data, but TVAE came closer.

TVAE's synthetic records were suspiciously close to real ones, hinting it may be copying rather than creating new data.

In [ ]:
!pip install -q sdv ctgan

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.19.0 requires botocore<1.36.4,>=1.36.0, but you have botocore 1.42.73 which is incompatible.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors

## 1. Load and Inspect the Data

In [ ]:
df = pd.read_csv("fraud_transactions.csv")
df.head()

,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,2/27/19 21:32,"fraud_Langosh, Wintheiser and Hyatt",food_dining,83.64,F,TX,"Physicist, medical",0
1,2/13/19 19:41,fraud_Dibbert and Sons,entertainment,79.13,M,PA,Secretary/administrator,0
2,1/11/19 20:03,"fraud_McDermott, Osinski and Morar",home,12.02,F,CA,"Buyer, industrial",0
3,1/20/19 9:08,fraud_Bauch-Raynor,grocery_pos,84.41,M,TN,Clothing/textile technologist,0
4,1/4/19 17:04,"fraud_Reichert, Huels and Hoppe",shopping_net,2.81,F,ME,Financial trader,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trans_date_trans_time  6486 non-null   object 
 1   merchant               6486 non-null   object 
 2   category               6486 non-null   object 
 3   amt                    6486 non-null   float64
 4   gender                 6486 non-null   object 
 5   state                  6486 non-null   object 
 6   job                    6486 non-null   object 
 7   is_fraud               6486 non-null   int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 405.5+ KB


In [ ]:
df["is_fraud"].value_counts(normalize=True)

is_fraud
0    0.925069
1    0.074931
Name: proportion, dtype: float64

## 2. Preprocessing

We drop high-cardinality columns (`merchant`, `job`, `state`) and the timestamp (`trans_date_trans_time`) which are not suitable for CTGAN/TVAE without heavy feature engineering.

- **Target:** `is_fraud`
- **Categorical:** `category`, `gender`
- **Numeric:** `amt`

In [ ]:
df = df.dropna().reset_index(drop=True)

target_col = "is_fraud"

cat_cols = ["category", "gender"]
num_cols = ["amt"]

model_df = df[cat_cols + num_cols + [target_col]].copy()
model_df[target_col] = model_df[target_col].astype(int)

print(model_df.shape)
model_df.head()

(6486, 4)


,category,gender,amt,is_fraud
0,food_dining,F,83.64,0
1,entertainment,M,79.13,0
2,home,F,12.02,0
3,grocery_pos,M,84.41,0
4,shopping_net,F,2.81,0


## 3. Train/Test Split on Real Data

In [ ]:
real_train, real_test = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42,
    stratify=model_df[target_col]
)

real_train.shape, real_test.shape

((5188, 4), (1298, 4))

## 4. Train CTGAN

In [ ]:
from ctgan import CTGAN

discrete_cols = cat_cols + [target_col]

ctgan = CTGAN(
    epochs=300,
    batch_size=500,
    verbose=True
)

ctgan.fit(model_df, discrete_columns=discrete_cols)

Gen. (-00.46) | Discrim. (-00.09): 100%|██████████| 300/300 [01:09<00:00,  4.29it/s]


## 5. Generate 5,000 Synthetic Records (CTGAN)

In [ ]:
n_synth = 5000
synthetic_ctgan = ctgan.sample(n_synth)
synthetic_ctgan.head()

,category,gender,amt,is_fraud
0,grocery_pos,F,226.611166,0
1,travel,F,3.627203,0
2,shopping_pos,M,881.678423,1
3,grocery_pos,F,3.262517,0
4,gas_transport,F,159.212607,0


In [ ]:
print("Real outcome distribution:")
print(model_df[target_col].value_counts(normalize=True))

print("\nCTGAN synthetic outcome distribution:")
print(synthetic_ctgan[target_col].value_counts(normalize=True))

Real outcome distribution:
is_fraud
0    0.925069
1    0.074931
Name: proportion, dtype: float64

CTGAN synthetic outcome distribution:
is_fraud
0    0.779
1    0.221
Name: proportion, dtype: float64


## 6. Train TVAE

In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(model_df)

for col in discrete_cols:
    metadata.update_column(column_name=col, sdtype='categorical')

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=300,
    batch_size=500
)

tvae.fit(model_df)

/opt/anaconda3/lib/python3.13/site-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/opt/anaconda3/lib/python3.13/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [ ]:
synthetic_tvae = tvae.sample(n_synth)

print("TVAE synthetic outcome distribution:")
print(synthetic_tvae[target_col].value_counts(normalize=True))

TVAE synthetic outcome distribution:
is_fraud
0    0.9506
1    0.0494
Name: proportion, dtype: float64


## 7. Evaluate Quality

### 7.1 Fidelity — KS Test (Numeric Features)

In [ ]:
def ks_per_feature(real_df, syn_df, num_cols):
    results = {}
    for col in num_cols:
        r = real_df[col].sample(min(5000, len(real_df)), random_state=42)
        s = syn_df[col].sample(min(5000, len(syn_df)), random_state=42)
        stat, pval = ks_2samp(r, s)
        results[col] = {"ks_stat": stat, "p_value": pval}
    return pd.DataFrame(results).T

ks_ctgan = ks_per_feature(model_df, synthetic_ctgan, num_cols)
ks_tvae  = ks_per_feature(model_df, synthetic_tvae, num_cols)

ks_compare = pd.DataFrame({
    "KS_CTGAN": ks_ctgan["ks_stat"],
    "KS_TVAE": ks_tvae["ks_stat"]
})

ks_compare

,KS_CTGAN,KS_TVAE
amt,0.0756,0.0382


### 7.2 Fidelity — Categorical Distribution Comparison

In [ ]:
for col in cat_cols + [target_col]:
    print(f"\n--- {col} ---")
    compare = pd.DataFrame({
        "Real": model_df[col].value_counts(normalize=True),
        "CTGAN": synthetic_ctgan[col].value_counts(normalize=True),
        "TVAE": synthetic_tvae[col].value_counts(normalize=True)
    }).fillna(0)
    print(compare)


--- category ---
                    Real   CTGAN    TVAE
category                                
entertainment   0.072155  0.0704  0.0980
food_dining     0.066759  0.0746  0.0836
gas_transport   0.106537  0.0992  0.0374
grocery_net     0.032994  0.0322  0.0140
grocery_pos     0.103916  0.1382  0.1152
health_fitness  0.062751  0.0472  0.0792
home            0.092815  0.0518  0.0922
kids_pets       0.087573  0.0794  0.1226
misc_net        0.050570  0.0664  0.0458
misc_pos        0.062751  0.0492  0.0656
personal_care   0.060284  0.0740  0.0362
shopping_net    0.080327  0.0894  0.1270
shopping_pos    0.089269  0.0850  0.0554
travel          0.031298  0.0430  0.0278

--- gender ---
          Real   CTGAN    TVAE
gender                        
F       0.5535  0.5284  0.5438
M       0.4465  0.4716  0.4562

--- is_fraud ---
              Real  CTGAN    TVAE
is_fraud                         
0         0.925069  0.779  0.9506
1         0.074931  0.221  0.0494


### 7.3 Utility — Train Synthetic, Test Real (TSTR)

In [ ]:
# Fit encoders on REAL training data only
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()

ohe.fit(real_train[cat_cols])
scaler.fit(real_train[num_cols])

def preprocess(df_subset):
    X_cat_enc = ohe.transform(df_subset[cat_cols])
    X_num_scaled = scaler.transform(df_subset[num_cols])
    X_all = np.hstack([X_cat_enc, X_num_scaled])
    y_out = df_subset[target_col].astype(int)
    return X_all, y_out

In [ ]:
# Baseline: Train REAL, Test REAL
X_real_train, y_real_train = preprocess(real_train)
X_real_test, y_real_test = preprocess(real_test)

rf_real = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_real.fit(X_real_train, y_real_train)

y_proba_real = rf_real.predict_proba(X_real_test)[:, 1]
y_pred_real = (y_proba_real >= 0.5).astype(int)

auc_real = roc_auc_score(y_real_test, y_proba_real)
f1_real = f1_score(y_real_test, y_pred_real)

print(f"TRTR — AUC: {auc_real:.4f}, F1: {f1_real:.4f}")

TRTR — AUC: 0.9932, F1: 0.8163


In [ ]:
# TSTR: CTGAN
X_ctgan, y_ctgan = preprocess(synthetic_ctgan)
rf_ctgan = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_ctgan.fit(X_ctgan, y_ctgan)

y_proba_ctgan = rf_ctgan.predict_proba(X_real_test)[:, 1]
y_pred_ctgan = (y_proba_ctgan >= 0.5).astype(int)

auc_ctgan = roc_auc_score(y_real_test, y_proba_ctgan)
f1_ctgan = f1_score(y_real_test, y_pred_ctgan)

# TSTR: TVAE
X_tvae, y_tvae = preprocess(synthetic_tvae)
rf_tvae = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_tvae.fit(X_tvae, y_tvae)

y_proba_tvae = rf_tvae.predict_proba(X_real_test)[:, 1]
y_pred_tvae = (y_proba_tvae >= 0.5).astype(int)

auc_tvae = roc_auc_score(y_real_test, y_proba_tvae)
f1_tvae = f1_score(y_real_test, y_pred_tvae)

utility_df = pd.DataFrame(
    {
        "AUC": [auc_real, auc_ctgan, auc_tvae],
        "F1":  [f1_real, f1_ctgan, f1_tvae]
    },
    index=[
        "Train REAL, Test REAL",
        "Train CTGAN, Test REAL",
        "Train TVAE, Test REAL"
    ]
)

utility_df

,AUC,F1
"Train REAL, Test REAL",0.993184,0.816327
"Train CTGAN, Test REAL",0.867155,0.439863
"Train TVAE, Test REAL",0.939578,0.590909


### 7.4 Privacy — Nearest-Neighbour Distance Check

In [ ]:
real_num = model_df[num_cols].sample(min(5000, len(model_df)), random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

# CTGAN
syn_ctgan_num = synthetic_ctgan[num_cols].sample(min(5000, len(synthetic_ctgan)), random_state=42)
dist_ctgan, _ = nn.kneighbors(syn_ctgan_num)
dist_ctgan = dist_ctgan.flatten()

# TVAE
syn_tvae_num = synthetic_tvae[num_cols].sample(min(5000, len(synthetic_tvae)), random_state=42)
dist_tvae, _ = nn.kneighbors(syn_tvae_num)
dist_tvae = dist_tvae.flatten()

privacy_df = pd.DataFrame({
    "CTGAN_dist": dist_ctgan,
    "TVAE_dist": dist_tvae
})

privacy_df.describe()

,CTGAN_dist,TVAE_dist
count,5.000000e+03,5000.000000
mean,6.110827e-01,0.135928
std,2.413562e+00,1.329325
min,6.585996e-07,0.000000
25%,5.903463e-03,0.000000
50%,1.823257e-02,0.010000
75%,9.239589e-02,0.030000
max,8.477558e+01,50.410000


### Interpretation

- **Fidelity:** Lower KS statistic = better distributional match. The categorical comparison shows how well each model preserves category proportions.
- **Utility:** TSTR AUC/F1 closer to the TRTR baseline = more useful synthetic data for downstream ML.
- **Privacy:** Higher nearest-neighbour distances = less memorization risk. Very small distances suggest the model may be copying real records.